# MinCED CRISPR Array Detection

![MinCED CRISPR Array Detection](https://proto-bio.github.io/proto-assets/images/tool/minced/hero.png)

This notebook demonstrates `run_minced`, which scans nucleotide sequences for the characteristic repeat-spacer architecture of CRISPR arrays. Given raw genomic DNA, MinCED reports each detected array with its repeat sequences, individual spacers, and their start positions. CRISPR detection is typically the first step when characterizing a CRISPR system in a newly sequenced microbe or identifying candidate loci for a Cas9 engineering pipeline.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("minced")
display_overview("minced")
display_docs_section("minced", "Background")

## Available tools

In [ ]:
display_available_tools("minced")

### `run_minced`

`run_minced` detects CRISPR arrays in nucleotide sequences using MinCED (Mining CRISPRs in Environmental Datasets). The tool scans for the repeat-spacer structure characteristic of CRISPR loci and returns each array with its repeat sequences, spacer sequences, and positions. The `min_num_repeats` and `min_repeat_length` thresholds control detection sensitivity.

In [ ]:
from proto_tools import (
    MincedInput,
    MincedConfig,
    run_minced,
)

In [ ]:
# Display input docs
display_api_reference("minced", "input", "run_minced")

from pathlib import Path

# Real Streptococcus pyogenes SF370 CRISPR1 array region (NC_002737.2).
crispr_sequence = "".join(l for l in Path("spyogenes_crispr_locus.fasta").read_text().splitlines() if not l.startswith(">"))
inputs = MincedInput(sequences=[crispr_sequence])

In [ ]:
# Display config docs
display_api_reference("minced", "config", "run_minced")

# Require at least 3 repeats of at least 27nt | see docs above for all fields
config = MincedConfig(
    min_num_repeats=3,
    min_repeat_length=27,
)

In [ ]:
# Run CRISPR array detection
result = run_minced(inputs, config)

In [ ]:
# Display output docs
display_api_reference("minced", "output", "run_minced")

# Report detected arrays, repeat counts, and the first few spacers per array
for seq_result in result.results:
    print(f"{seq_result.sequence_id}: {seq_result.num_arrays} CRISPR array(s) found")
    if seq_result.has_crispr:
        for i, array in enumerate(seq_result.crispr_arrays):
            print(f"  Array {i+1}: {array.num_repeats} repeats")
            print(f"  Spacers: {array.spacers[:3]}..." if len(array.spacers) > 3 else f"  Spacers: {array.spacers}")

## Export Results

CRISPR detection results can be exported to CSV or JSON. Both formats flatten to one row per repeat-spacer unit, tagged with its `sequence_id` and `array_index`.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="minced_results", export_path=output_dir, file_format="json")
print(f"CRISPR detection results exported to {output_dir}")